# Análisis de precios y marcas (listado completo de motos)

Fuente: `data/processed/listados_crudos/motos_listado.json` (2.939 anuncios).
Filtro: moneda USD y 100 < precio < 5000. Sin gráficos.

In [1]:
import json
import re
import pandas as pd
from pathlib import Path

BASE = None
for nivel in ("data/processed", "../data/processed", "../../data/processed"):
    if Path(nivel).exists():
        BASE = Path(nivel)
        break
assert BASE is not None

with open(BASE / "listados_crudos/motos_listado.json", encoding="utf-8") as f:
    df = pd.DataFrame(json.load(f)["anuncios"])
print("Anuncios crudos:", len(df))

Anuncios crudos: 2939


In [2]:
total_inicial = len(df)

solo_usd = df["moneda"] == "USD"
df = df[solo_usd]
n_no_usd = int((~solo_usd).sum())
print(f"Eliminados por no USD: {n_no_usd}")

antes_rango = len(df)
df = df[df["precio"] > 100]
n_le_100 = int(antes_rango - len(df))
antes_rango = len(df)
df = df[df["precio"] < 5000]
n_ge_5000 = int(antes_rango - len(df))
print(f"Eliminados por precio <= 100: {n_le_100}")
print(f"Eliminados por precio >= 5000: {n_ge_5000}")

print("Total tras filtrar:", len(df))

Eliminados por no USD: 124
Eliminados por precio <= 100: 115
Eliminados por precio >= 5000: 200
Total tras filtrar: 2500


In [3]:
MARCAS = [
    ("topmaq", "Topmaq"),
    ("vedca", "Vedca"),
    ("\\bizuki\\b", "Izuki"),
    ("yoazaky|yoazaki|yoa ?zaky|yoazak", "Yoazaki"),
    ("mountain", "Mountain"),
    ("bucatti|bucati|buccati", "Bucatti"),
    ("fly racing|flywing|\\bfly\\b", "Fly Racing"),
    ("xmox", "Xmox"),
    ("mishozuki", "Mishozuki"),
    ("\\bava\\b", "Ava"),
    ("halcon", "Halcon"),
    ("kvitova|kivitova|kitova", "Kvitova"),
    ("murasaki", "Murasaki"),
    ("panda", "Panda"),
    ("\\bmaf\\b", "Maf"),
    ("\\bmaox\\b", "Maox"),
    ("\\btorvan\\b", "Torvan"),
    ("mikazuki", "Mikazuki"),
    ("dcmlz", "DCM-LZ"),
    ("protenzza|protenza|prontenza", "Protenzza"),
    ("voltax", "Voltax"),
    ("volta", "Volta"),
    ("meacon", "Meacon"),
    ("zuzuki|suzuki|\\bzuki\\b", "Zuki"),
    ("\\bsnopy\\b", "Snopy"),
    ("yadea", "Yadea"),
    ("\\bxtreme\\b", "Xtreme"),
    ("mavis", "Mavis"),
    ("\\bgt\\b", "GT"),
]

def _norm(s):
    plano = re.sub(r"[^a-z0-9]+", " ", (s or "").lower())
    return re.sub(r"\s+", " ", plano).strip()

def marca_de_titulo(titulo):
    texto = _norm(titulo)
    for patron, marca in MARCAS:
        if re.search(patron, texto):
            return marca
    return None

df["marca"] = df["titulo"].map(marca_de_titulo)

In [4]:
print("Distribución de precios tras filtrar (describe):")
print(df["precio"].describe().to_string())
print()
pct = df["precio"].quantile([0.10, 0.25, 0.50, 0.75, 0.90])
print("Percentiles:")
print(pct.to_string())

Distribución de precios tras filtrar (describe):
count    2500.000000
mean     2047.239996
std      1345.308034
min       110.000000
25%       850.000000
50%      1715.000000
75%      2950.000000
max      4990.000000

Percentiles:
0.10     550.0
0.25     850.0
0.50    1715.0
0.75    2950.0
0.90    4350.0


In [5]:
top10 = df["marca"].value_counts().head(10)
print("Top 10 marcas por número de anuncios:")
print(top10.to_string())

print("\nPrecio mediano y media por marca (top 10):")
top10_idx = top10.index
tabla = df[df["marca"].isin(top10_idx)].groupby("marca")["precio"].agg(["median", "mean"]).sort_values("median", ascending=False)
print(tabla.to_string())

Top 10 marcas por número de anuncios:
marca
Topmaq        265
Vedca         107
Yoazaki        86
Mountain       81
Kvitova        70
Maf            59
Bucatti        54
Mishozuki      50
Fly Racing     34
Izuki          31

Precio mediano y media por marca (top 10):
            median         mean
marca                          
Kvitova     3500.0  2972.571429
Izuki       2800.0  2891.935484
Topmaq      2650.0  2977.698113
Mountain    2600.0  2414.567901
Fly Racing  2450.0  2337.058824
Yoazaki     1950.0  2178.372093
Mishozuki   1925.0  2885.100000
Bucatti     1900.0  2014.444444
Vedca       1720.0  2021.028037
Maf         1150.0  2127.627119


In [6]:
import sys
k = Path("src/scrapers") if Path("src/scrapers/revolico.py").exists() else Path("../src/scrapers")
sys.path.insert(0, str(k.resolve()))
import revolico as rv

listados = sorted((rv.CACHE_DIR / "motos").glob("listado_*.html"))
cat = rv.extraer_catalogos(listados[0].read_text(encoding="utf-8"))
df["municipio"] = df["municipalityId"].map(cat["municipios"])

top_muni = df["municipio"].value_counts().head(10)
print("Precio mediano por municipio (top 10 municipios):")
top_muni_idx = top_muni.index
df[df["municipio"].isin(top_muni_idx)].groupby("municipio")["precio"].median().sort_values(ascending=False)

Precio mediano por municipio (top 10 municipios):


municipio
Cerro              1890.0
Centro Habana      1800.0
Playa              1750.0
Marianao           1750.0
Arroyo Naranjo     1700.0
Plaza              1640.0
Diez de Octubre    1620.0
La Lisa            1600.0
Boyeros            1550.0
Habana Vieja       1410.0
Name: precio, dtype: float64

In [7]:
con_marca = df["marca"].notna().sum()
sin_marca = df["marca"].isna().sum()
print("Anuncios con marca detectada:", con_marca)
print("Anuncios sin marca detectada:", sin_marca)

Anuncios con marca detectada: 981
Anuncios sin marca detectada: 1519
